##Silver Fraud Indicators
### Goal

##### Create:
- silver_fraud_indicators
- quarantine_invalid_fraud_indicators

##### Validate:

- claim_id exists in silver_claims
- risk_score between 0 and 100
- clean boolean flags
- remove invalid records

In [0]:
from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"

In [0]:
fraud_bronze = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_fraud_indicators")

claims = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims") \
    .select("claim_id") \
    .dropDuplicates()

### Clean Fraud Data
#### What this does:
- trims claim_id
- converts types
- prepares risk_score validation

In [0]:
fraud_prepared = (
    fraud_bronze
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("previous_claims_count", F.col("previous_claims_count").cast("int"))
    .withColumn("suspicious_amount_flag", F.col("suspicious_amount_flag").cast("boolean"))
    .withColumn("duplicate_claim_flag", F.col("duplicate_claim_flag").cast("boolean"))
    .withColumn("late_report_flag", F.col("late_report_flag").cast("boolean"))
    .withColumn("high_risk_region_flag", F.col("high_risk_region_flag").cast("boolean"))
    .withColumn("risk_score", F.col("risk_score").cast("int"))
)

In [0]:
fraud_joined = (
    fraud_prepared.alias("f")
    .join(claims.alias("c"), "claim_id", "left")
    .withColumn("claim_exists", F.col("c.claim_id").isNotNull())
)

###Invalid Fraud Indicators (Quarantine)
#### What this does:
- captures bad records
- missing claim_id
- invalid risk_score

In [0]:
invalid_fraud = (
    fraud_joined
    .filter(
        F.col("claim_id").isNull()
        | (~F.col("claim_exists"))
        | (F.col("risk_score") < 0)
        | (F.col("risk_score") > 100)
    )
    .withColumn("record_id", F.col("claim_id"))
    .withColumn("source_table", F.lit("bronze_fraud_indicators"))
    .withColumn("error_reason", F.lit("fraud_validation_failed"))
    .withColumn("error_severity", F.lit("HIGH"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn(
        "original_record_json",
        F.to_json(F.struct(*[F.col(c) for c in fraud_prepared.columns]))
    )
)

###Valid Fraud Indicators (Silver)
#### What this does:
- keeps only trusted fraud data
- ensures risk_score is valid
- removes duplicates

In [0]:
silver_fraud = (
    fraud_joined
    .filter(
        F.col("claim_id").isNotNull()
        & F.col("claim_exists")
        & F.col("risk_score").between(0, 100)
    )
    .drop("claim_exists")
    .dropDuplicates(["claim_id"])
)

In [0]:
silver_fraud.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_fraud_indicators")

In [0]:
(
    invalid_fraud
    .select(
        "record_id",
        "source_table",
        "error_reason",
        "error_severity",
        "quarantine_timestamp"
    )
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_fraud_indicators")
)

In [0]:
print("Silver fraud:", spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_fraud_indicators").count())
print("Quarantine fraud:", spark.table(f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_fraud_indicators").count())